# NB-04 — Pydantic Validation

**Learning goal:** The tool call gives you a Python dict. Pydantic turns that dict into a validated, typed object — and raises a clear error if anything is wrong. This is the safety net between the LLM and your application logic.

**Concepts covered**
- `BaseModel` definition with `Literal` type constraints
- `@field_validator` for custom business rules
- `ValidationError` — catching it and extracting which fields failed
- The retry pattern — re-prompting when validation fails
- Why Pydantic and function calling work together

## Cell 1 — Setup

In [1]:
from pydantic import BaseModel, field_validator, ValidationError
from typing import Literal
import json

## Cell 2 — Define IntakeRecord

In [2]:
class IntakeRecord(BaseModel):
    query_type: Literal[
        "complaint", "submission", "variation", "safety_signal",
        "label_update", "inspection", "general_enquiry"
    ]
    regulation_ref: Literal[
        "FDA_21CFR", "EMA_CTR", "ICH_E2A", "ICH_Q10",
        "CDSCO_NDC", "GxP_GMP", "GxP_GCP", "other"
    ]
    product_area: Literal[
        "oncology", "cardiovascular", "infectious_disease",
        "cmc", "clinical", "labelling", "general"
    ]
    urgency: Literal["routine", "standard", "urgent", "critical"]
    submitting_team: str

    @field_validator("submitting_team")
    @classmethod
    def team_must_not_be_empty(cls, v):
        if not v.strip():
            raise ValueError("submitting_team cannot be empty")
        return v.strip().title()

    @field_validator("submitting_team")
    @classmethod
    def team_not_a_person(cls, v):
        # Heuristic: a single capitalised alphabetic word is likely a person's name
        if len(v.split()) == 1 and v[0].isupper() and v.isalpha():
            raise ValueError(
                "submitting_team must be a team or function, not an individual name. "
                "Examples: 'PV Team', 'CMC Regulatory', 'Clinical Operations'."
            )
        return v

## Cell 3 — Valid Record

In [3]:
valid_data = {
    "query_type": "safety_signal",
    "regulation_ref": "ICH_E2A",
    "product_area": "clinical",
    "urgency": "critical",
    "submitting_team": "PV Team"
}

record = IntakeRecord(**valid_data)
print(record)
print(record.model_dump())

query_type='safety_signal' regulation_ref='ICH_E2A' product_area='clinical' urgency='critical' submitting_team='Pv Team'
{'query_type': 'safety_signal', 'regulation_ref': 'ICH_E2A', 'product_area': 'clinical', 'urgency': 'critical', 'submitting_team': 'Pv Team'}


## Cell 4 — Invalid Enum Value

In [ ]:
try:
    bad_enum = IntakeRecord(
        query_type="complaint",
        regulation_ref="WHO_GMP",        # not in the enum
        product_area="clinical",
        urgency="critical",
        submitting_team="PV Team"
    )
except ValidationError as e:
    print("Validation failed:")
    for err in e.errors():
        print(f"  Field: {err['loc'][0]}  |  Error: {err['msg']}")

## Cell 5 — Individual Name as Team

In [ ]:
try:
    bad_team = IntakeRecord(
        query_type="submission",
        regulation_ref="FDA_21CFR",
        product_area="cmc",
        urgency="standard",
        submitting_team="Rahul"          # single name — should fail
    )
except ValidationError as e:
    print("Validation failed:")
    for err in e.errors():
        print(f"  Field: {err['loc'][0]}  |  Error: {err['msg']}")

## Cell 6 — Extracting Missing Fields for Re-Prompting

In [ ]:
def get_missing_fields(validation_error: ValidationError) -> list[str]:
    """Return the list of field names that failed validation."""
    return [str(err["loc"][0]) for err in validation_error.errors()]

try:
    IntakeRecord(
        query_type="submission",
        regulation_ref="FDA_21CFR",
        product_area="cmc",
        urgency="standard",
        submitting_team=""               # empty
    )
except ValidationError as e:
    missing = get_missing_fields(e)
    print(f"Missing or invalid fields: {missing}")
    print(f"Ask the user for: {', '.join(missing)}")

## Cell 7 — The Retry Pattern

In [ ]:
def validate_with_retry(tool_args: dict, max_attempts: int = 2):
    for attempt in range(max_attempts):
        try:
            record = IntakeRecord(**tool_args)
            print(f"Validation passed on attempt {attempt + 1}")
            return record
        except ValidationError as e:
            missing = get_missing_fields(e)
            print(f"Attempt {attempt + 1} failed. Missing: {missing}")
            if attempt < max_attempts - 1:
                print("  → Would re-prompt user for missing fields")
            else:
                print("  → Max attempts reached. Falling back to manual collection.")
                return None

# Simulate a partial extraction
validate_with_retry({
    "query_type": "submission",
    "regulation_ref": "FDA_21CFR",
    "product_area": "cmc",
    "urgency": "standard",
    "submitting_team": ""
})

### Key Takeaway

Pydantic is the contract between the LLM and your code. Function calling makes Claude return structured data; Pydantic guarantees that data meets your business rules. Together they eliminate silent failures — every validation error is explicit and actionable.